In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    DecimalType,
    DateType,
    TimestampType,
    BooleanType
)

CUSTOMERS_PATH = (
    "/Volumes/credlake/landing/raw/"
    "customers/snapshot_date=2026-08-01/"
)

SNAPSHOT_DATE = "2026-08-01"

customer_schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("customer_type", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("document_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("state", StringType(), True),
    StructField("risk_rating", StringType(), True),
    StructField("monthly_income", DecimalType(18, 2), True),
    StructField("created_at", DateType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_batch_id", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
customers_source_df = (
    spark.read
    .format("json")
    .schema(customer_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(CUSTOMERS_PATH)
)

In [0]:
customers_source_df.printSchema()

display(customers_source_df.limit(10))

In [0]:
customers_bronze_df = (
    customers_source_df
    .select(
        "customer_id",
        "customer_type",
        "customer_name",
        "document_id",
        "email",
        "state",
        "risk_rating",
        "monthly_income",
        "created_at",
        "updated_at",
        "is_active",
        "source_system",
        "source_batch_id",
        F.to_date(F.lit(SNAPSHOT_DATE)).alias("snapshot_date"),
        F.col("_metadata.file_path").alias("source_file_path"),
        F.col("_metadata.file_name").alias("source_file_name"),
        F.col("_metadata.file_modification_time")
            .alias("source_file_modification_time"),
        F.col("_corrupt_record").alias("corrupt_record"),
        F.current_timestamp().alias("ingested_at")
    )
)

In [0]:
customer_metrics_df = customers_bronze_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("distinct_customer_ids"),
    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("null_customer_ids"),
    F.sum(
        F.when(F.col("corrupt_record").isNotNull(), 1).otherwise(0)
    ).alias("corrupt_records"),
    F.countDistinct("source_file_path").alias("source_files")
)

display(customer_metrics_df)

In [0]:
metrics = customer_metrics_df.first()

assert metrics["total_rows"] == 2000, (
    f"Esperados 2000 clientes, encontrados {metrics['total_rows']}"
)

assert metrics["distinct_customer_ids"] == 2000, (
    "Foram encontrados customer_id duplicados"
)

assert metrics["null_customer_ids"] == 0, (
    "Foram encontrados customer_id nulos"
)

assert metrics["corrupt_records"] == 0, (
    "Foram encontrados registros JSON corrompidos"
)

print("Validações da origem concluídas com sucesso.")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS credlake.bronze.customers_raw (
    customer_id                        BIGINT,
    customer_type                      STRING,
    customer_name                      STRING,
    document_id                        STRING,
    email                              STRING,
    state                              STRING,
    risk_rating                        STRING,
    monthly_income                     DECIMAL(18,2),
    created_at                         DATE,
    updated_at                         TIMESTAMP,
    is_active                          BOOLEAN,
    source_system                      STRING,
    source_batch_id                    STRING,
    snapshot_date                      DATE,
    source_file_path                   STRING,
    source_file_name                   STRING,
    source_file_modification_time      TIMESTAMP,
    corrupt_record                     STRING,
    ingested_at                        TIMESTAMP
)
USING DELTA
COMMENT 'Snapshots de clientes recebidos da origem, preservados na camada Bronze.'
TBLPROPERTIES (
    'data_layer' = 'bronze',
    'data_domain' = 'customer',
    'contains_synthetic_data' = 'true'
)
""")

In [0]:
(
    customers_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "replaceWhere",
        f"snapshot_date = DATE '{SNAPSHOT_DATE}'"
    )
    .saveAsTable("credlake.bronze.customers_raw")
)

In [0]:
display(
    spark.sql("""
        SELECT
            snapshot_date,
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS distinct_customers,
            COUNT(DISTINCT source_file_path) AS source_files,
            SUM(CASE WHEN corrupt_record IS NOT NULL THEN 1 ELSE 0 END)
                AS corrupt_records,
            MIN(ingested_at) AS first_ingestion,
            MAX(ingested_at) AS last_ingestion
        FROM credlake.bronze.customers_raw
        GROUP BY snapshot_date
        ORDER BY snapshot_date
    """)
)